# 1. Data Quality Assessment (Diagnóstico de Calidad)

## Objetivo
Evaluar la calidad de los datos crudos recopilados. Identificar problemas como:
- Valores faltantes o nulos
- Tipos de datos incorrectos
- Rangos inválidos
- Duplicados
- Inconsistencias en el formato

## Salida
Reporte detallado de problemas encontrados para guiar la limpieza

## 1.1 Importar librerías necesarias

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('✅ Librerías importadas exitosamente')

## 1.2 Cargar datos crudos

In [ ]:
# Define la ruta de los datos crudos
DATA_RAW_PATH = '../../data/00_raw/traffic_data.csv'

# Carga el CSV
df_raw = pd.read_csv(DATA_RAW_PATH)

print(f'✅ Datos cargados: {len(df_raw)} registros')
print(f'📊 Dimensiones: {df_raw.shape[0]} filas × {df_raw.shape[1]} columnas')
print(f'\n📋 Columnas presentes:')
print(df_raw.columns.tolist())

## 1.3 Vista general de los datos

In [ ]:
# Primeras filas
print('PRIMERAS 5 FILAS:')
print(df_raw.head())
print('\n' + '='*80 + '\n')

# Tipos de datos
print('TIPOS DE DATOS:')
print(df_raw.dtypes)
print('\n' + '='*80 + '\n')

# Info general
print('INFO GENERAL:')
df_raw.info()

## 1.4 Análisis de valores nulos/faltantes

In [ ]:
# Conteo de valores nulos por columna
print('VALORES NULOS/FALTANTES:')
valores_nulos = df_raw.isnull().sum()
porcentaje_nulos = (df_raw.isnull().sum() / len(df_raw) * 100).round(2)

tabla_nulos = pd.DataFrame({
    'Columna': df_raw.columns,
    'Nulos': valores_nulos.values,
    'Porcentaje': porcentaje_nulos.values
})

print(tabla_nulos.to_string(index=False))
print(f'\n⚠️  Total de celdas con datos faltantes: {df_raw.isnull().sum().sum()} / {df_raw.size}')

## 1.5 Análisis de duplicados

In [ ]:
# Detectar duplicados
duplicados_totales = df_raw.duplicated().sum()
duplicados_por_timestamp = df_raw.duplicated(subset=['timestamp', 'avenida']).sum()

print('ANÁLISIS DE DUPLICADOS:')
print(f'Duplicados totales (filas idénticas): {duplicados_totales}')
print(f'Duplicados por timestamp+avenida: {duplicados_por_timestamp}')

if duplicados_totales > 0:
    print('\n🔴 ADVERTENCIA: Se encontraron filas duplicadas')
    print('Ejemplo de duplicados:')
    duplicados_mask = df_raw.duplicated(keep=False)
    print(df_raw[duplicados_mask].sort_values('timestamp').head(10))
else:
    print('\n✅ No se encontraron duplicados')

## 1.6 Análisis de rangos válidos

In [ ]:
# Define rangos válidos para cada variable
rangos_validos = {
    'velocidad': (0, 120),
    'densidad': (1, 3),
    'detenciones': (0, 30),
    'latitud': (-90, 90),
    'longitud': (-180, 180)
}

print('ANÁLISIS DE RANGOS VÁLIDOS:\n')

problemas_rango = {}

for columna, (minimo, maximo) in rangos_validos.items():
    if columna in df_raw.columns:
        # Valores fuera de rango
        fuera_rango = ((df_raw[columna] < minimo) | (df_raw[columna] > maximo)).sum()
        
        # Estadísticas
        media = df_raw[columna].mean()
        minimo_actual = df_raw[columna].min()
        maximo_actual = df_raw[columna].max()
        
        print(f'{columna}:')
        print(f'  Rango esperado: [{minimo}, {maximo}]')
        print(f'  Rango actual: [{minimo_actual}, {maximo_actual}]')
        print(f'  Media: {media:.2f}')
        print(f'  Valores fuera de rango: {fuera_rango}')
        
        if fuera_rango > 0:
            print(f'  🔴 ADVERTENCIA: {fuera_rango} valores inválidos')
            problemas_rango[columna] = fuera_rango
        else:
            print(f'  ✅ OK')
        print()

## 1.7 Validación de columnas categóricas

In [ ]:
print('ANÁLISIS DE COLUMNAS CATEGÓRICAS:\n')

# Avenidas únicas
if 'avenida' in df_raw.columns:
    avenidas_unicas = df_raw['avenida'].unique()
    print(f'Avenidas encontradas ({len(avenidas_unicas)}):')  
    for av in sorted(avenidas_unicas):
        cuenta = (df_raw['avenida'] == av).sum()
        print(f'  - {av}: {cuenta} registros')
    print()

# Descripciones
if 'descripcion' in df_raw.columns:
    descripciones_vacias = (df_raw['descripcion'].isna().sum() + 
                           (df_raw['descripcion'].str.strip() == '').sum())
    print(f'Descripciones vacías: {descripciones_vacias}')
    print(f'Descripciones válidas: {len(df_raw) - descripciones_vacias}')

## 1.8 Análisis temporal

In [ ]:
print('ANÁLISIS TEMPORAL:\n')

if 'timestamp' in df_raw.columns:
    # Intentar convertir a datetime
    try:
        df_raw['timestamp'] = pd.to_datetime(df_raw['timestamp'])
        print(f'✅ Timestamps válidos')
        print(f'Rango temporal: {df_raw["timestamp"].min()} a {df_raw["timestamp"].max()}')
        print(f'Duración: {(df_raw["timestamp"].max() - df_raw["timestamp"].min()).days} días')
    except:
        print('🔴 ERROR: No se pudieron convertir timestamps a datetime')
        print('Valores de ejemplo:')
        print(df_raw['timestamp'].head())

## 1.9 Estadísticas descriptivas

In [ ]:
print('ESTADÍSTICAS DESCRIPTIVAS (Variables numéricas):\n')
print(df_raw.describe().round(2))

## 1.10 Detección de outliers

In [ ]:
# Método IQR para detectar outliers
def detectar_outliers(df, columna):
    Q1 = df[columna].quantile(0.25)
    Q3 = df[columna].quantile(0.75)
    IQR = Q3 - Q1
    limite_inf = Q1 - 1.5 * IQR
    limite_sup = Q3 + 1.5 * IQR
    
    outliers = df[(df[columna] < limite_inf) | (df[columna] > limite_sup)]
    return len(outliers), limite_inf, limite_sup

print('DETECCIÓN DE OUTLIERS (Método IQR):\n')

columnas_numericas = ['velocidad', 'densidad', 'detenciones']

for col in columnas_numericas:
    if col in df_raw.columns:
        outliers, limite_inf, limite_sup = detectar_outliers(df_raw, col)
        print(f'{col}:')
        print(f'  Límites válidos: [{limite_inf:.2f}, {limite_sup:.2f}]')
        print(f'  Outliers encontrados: {outliers}')
        if outliers > 0:
            print(f'  ⚠️  {(outliers/len(df_raw)*100):.2f}% de los datos')
        print()

## 1.11 Resumen de hallazgos

In [ ]:
print('='*80)
print('RESUMEN DE HALLAZGOS DE CALIDAD')
print('='*80)

hallazgos = {
    '✅ Total de registros': len(df_raw),
    '✅ Total de columnas': len(df_raw.columns),
    '⚠️  Valores nulos totales': df_raw.isnull().sum().sum(),
    '⚠️  Duplicados encontrados': df_raw.duplicated().sum(),
    '⚠️  Valores fuera de rango': sum(problemas_rango.values()) if problemas_rango else 0,
}

for hallazgo, valor in hallazgos.items():
    print(f'{hallazgo}: {valor}')

print('\n' + '='*80)
print('RECOMENDACIONES PARA LIMPIEZA:')
print('='*80)

if df_raw.isnull().sum().sum() > 0:
    print('1. ⚠️  VALORES NULOS: Imputar o eliminar filas según estrategia')
    
if df_raw.duplicated().sum() > 0:
    print('2. ⚠️  DUPLICADOS: Remover filas duplicadas')
    
if problemas_rango:
    print('3. ⚠️  RANGOS INVÁLIDOS: Investigar y corregir valores fuera de rango')
    
print('4. ✅ Convertir timestamp a datetime')
print('5. ✅ Normalizar variables numéricas (si es necesario para análisis)')
print('\n✅ Próximo paso: Ejecutar Data_Cleaning.ipynb')